# DialectSentEval 2026, Subtask 2 - Development

Team **Sabaa**. This notebook trains the system and evaluates it on the
development split, where the source polarity is supplied.

The system generates several candidate rewrites from two Arabic
sequence-to-sequence models, then selects among them with a scoring function
built from the task's own evaluation signals. Runtime on Kaggle T4 x2 is about
two and a half hours.

Requires `SentimentSwapSharedTaskTrain.xlsx` and
`SentimentSwapSharedTaskVal.xlsx` attached as a Kaggle dataset.

## 1. Environment

In [ ]:
%%capture
!pip install -q transformers==4.44.0 datasets sacrebleu sentencepiece accelerate openpyxl

## 2. Configuration

Every hyperparameter used in the run is set here and nowhere else.

In [ ]:
"""Imports, file discovery, and the complete run configuration."""

import os
import gc
import glob
import json
import zipfile
import warnings

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sacrebleu.metrics import BLEU, CHRF
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

warnings.filterwarnings("ignore")


def find_file(filename):
    """Return the path to `filename`, searching the usual Kaggle and local roots.

    Direct paths are checked first because they are cheap; a recursive search is
    used only as a fallback. Raising with the list of visible spreadsheets makes
    a misplaced dataset easy to diagnose.
    """
    for path in (f"/kaggle/input/{filename}", f"/workspace/{filename}",
                 f"/root/{filename}", f"./{filename}"):
        if os.path.exists(path):
            return path

    for root in ("/kaggle/input", "/workspace", "/root", "."):
        matches = glob.glob(f"{root}/**/{filename}", recursive=True)
        if matches:
            return matches[0]

    visible = glob.glob("/kaggle/**/*.xlsx", recursive=True)
    raise FileNotFoundError(f"{filename} not found. Visible .xlsx files: {visible[:8]}")


WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "/workspace"

# Models. Two Arabic encoder-decoders supply the candidate pool; the classifier
# is the one the shared task uses for evaluation, so reranking optimises the
# same signal the task grades.
ARAT5_MODEL   = "UBC-NLP/AraT5v2-base-1024"
ARABART_MODEL = "moussaKam/AraBART"
CLF_MODEL     = "CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment"

ARAT5_DIR   = f"{WORK_DIR}/arat5v2"
ARABART_DIR = f"{WORK_DIR}/arabart"

# Training. Identical for both generators so the only difference between the
# two candidate streams is the pretrained model behind it.
MAX_LEN     = 128
TRAIN_BATCH = 4
EVAL_BATCH  = 8
GRAD_ACCUM  = 2
EPOCHS      = 15
LEARNING_RATE = 5e-5
WARMUP_STEPS  = 500
WEIGHT_DECAY  = 0.01

SEED   = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.cuda.empty_cache()

# Decoding and reranking, development configuration.
GEN_BATCH  = 8
CLF_BATCH  = 64
NUM_BEAMS  = 5   # per model, so the pooled candidate set holds 10
NUM_RETURN = 5

W_SENTIMENT = 0.60
W_CONSENSUS = 0.20
W_CONTENT   = 0.20

print(f"Device : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
print(f"Output : {WORK_DIR}")

## 3. Data

The training set is doubled by reading each pair in reverse. This is sound
because a sentiment swap is symmetric, and it has the side benefit of balancing
the two polarity classes exactly.

In [ ]:
def invert_polarity(polarity):
    """Return the opposite polarity label."""
    return "Negative" if polarity == "Positive" else "Positive"


def augment_symmetrically(df):
    """Double the training set by reading every pair in the reverse direction.

    A sentiment swap is symmetric: if `target` is the polarity-inverted rewrite
    of `source`, then `source` is equally a valid rewrite of `target` under the
    opposite polarity. Adding the reversed pair costs no annotation, and because
    each added pair carries the inverted label it also balances the two classes.
    """
    reversed_pairs = df.copy()
    reversed_pairs["source"]          = df["target"]
    reversed_pairs["target"]          = df["source"]
    reversed_pairs["source_polarity"] = df["source_polarity"].apply(invert_polarity)
    reversed_pairs["id"]              = df["id"] + 100_000  # keep ids traceable

    combined = pd.concat([df, reversed_pairs], ignore_index=True)
    return combined.sample(frac=1, random_state=SEED).reset_index(drop=True)


def build_prompt(row):
    """Format one instance as an Arabic instruction.

    The instruction is Arabic because both generators were pretrained on Arabic
    only; an English frame would sit outside their pretraining distribution.
    """
    src_pol = row["source_polarity"]
    tgt_pol = invert_polarity(src_pol)
    return f"اعكس قطبية النص من {src_pol} إلى {tgt_pol}: {row['source']}"


TRAIN_PATH = find_file("SentimentSwapSharedTaskTrain.xlsx")
VAL_PATH   = find_file("SentimentSwapSharedTaskVal.xlsx")

train_raw = pd.read_excel(TRAIN_PATH)
val_df    = pd.read_excel(VAL_PATH)

train_df = augment_symmetrically(train_raw)

train_df["input_text"]  = train_df.apply(build_prompt, axis=1)
train_df["target_text"] = train_df["target"].astype(str)

# The development split supplies source_polarity, so the swap direction is known.
val_df["target_polarity"] = val_df["source_polarity"].apply(invert_polarity)
val_df["input_text"]      = val_df.apply(build_prompt, axis=1)

print(f"Training pairs : {len(train_raw)} -> {len(train_df)} after augmentation")
print(f"Class balance  : {train_df['source_polarity'].value_counts().to_dict()}")
print(f"Development    : {len(val_df)}")
print(f"Example prompt : {train_df['input_text'].iloc[0][:78]}")

## 4. Sentiment classifier

CAMeLBERT-DA is the classifier the shared task evaluates with, so using it to
rank candidates means optimising the quantity we are graded on rather than a
proxy for it.

In [ ]:
def load_sentiment_classifier():
    """Load CAMeLBERT-DA, working around a version-dependent loading guard.

    The checkpoint ships in the legacy .bin format. transformers >= 5 refuses to
    torch.load such files on PyTorch < 2.6 (CVE-2025-32434), while older
    transformers has no such guard at all. The check is therefore neutralised
    only when it exists, and restored immediately afterwards.
    """
    import transformers.modeling_utils as modeling_utils

    guard = getattr(modeling_utils, "check_torch_load_is_safe", None)
    if guard is not None:
        modeling_utils.check_torch_load_is_safe = lambda: None
    try:
        tokenizer = AutoTokenizer.from_pretrained(CLF_MODEL)
        model = AutoModelForSequenceClassification.from_pretrained(
            CLF_MODEL, torch_dtype=torch.float32
        ).to(DEVICE)
    finally:
        if guard is not None:
            modeling_utils.check_torch_load_is_safe = guard

    model.eval()
    return tokenizer, model


clf_tokenizer, clf_model = load_sentiment_classifier()
LABEL_MAP    = clf_model.config.id2label                    # e.g. {0: 'positive', ...}
LABEL_TO_IDX = {v.capitalize(): k for k, v in LABEL_MAP.items()}


def classify(texts):
    """Return the predicted label and its confidence for each text.

    Labels come back lower-cased, so callers capitalise before comparing them
    against the Positive/Negative strings used throughout this notebook.
    """
    encoded = clf_tokenizer(
        texts, return_tensors="pt", truncation=True, max_length=128, padding=True
    ).to(DEVICE)
    with torch.no_grad():
        probs = torch.softmax(clf_model(**encoded).logits, dim=-1).cpu().numpy()
    return [
        {"label": LABEL_MAP[int(np.argmax(p))], "score": float(np.max(p))}
        for p in probs
    ]


def classify_all(texts):
    """Run `classify` over a long list in classifier-sized batches."""
    results = []
    for start in range(0, len(texts), CLF_BATCH):
        results.extend(classify(texts[start:start + CLF_BATCH]))
    return results


probe = classify(["برنامج رائع جداً", "برنامج سيء جداً"])
assert probe[0]["label"].capitalize() == "Positive"
assert probe[1]["label"].capitalize() == "Negative"
print("Classifier ready. Labels:", LABEL_MAP)

## 5. Training

Both generators are trained by the same function with the same settings, so the
only thing separating the two candidate streams is the pretrained model.

In [ ]:
def train_generator(model_name, save_dir, train_df):
    """Fine-tune one seq2seq model and write it to `save_dir`.

    save_strategy is "no" on purpose. The Trainer's periodic checkpointing
    serialises through safetensors, which rejects non-contiguous tensors and
    fails mid-run under fp16; saving once at the end after an explicit
    .contiguous() pass avoids that failure entirely.

    Label smoothing is left off. It lowers training loss but flattens the output
    distribution, and this system ranks candidates rather than taking the single
    best one, so a flatter distribution gives the reranker less to separate.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize(batch):
        model_inputs = tokenizer(
            batch["input_text"], max_length=MAX_LEN, truncation=True, padding=False
        )
        labels = tokenizer(
            batch["target_text"], max_length=MAX_LEN, truncation=True, padding=False
        )
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    dataset = Dataset.from_pandas(train_df[["input_text", "target_text"]])
    tokenized = dataset.map(tokenize, batched=True, remove_columns=dataset.column_names)

    gc.collect()
    torch.cuda.empty_cache()
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    args = Seq2SeqTrainingArguments(
        output_dir=save_dir,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=TRAIN_BATCH,
        per_device_eval_batch_size=EVAL_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        warmup_steps=WARMUP_STEPS,
        weight_decay=WEIGHT_DECAY,
        save_strategy="no",
        predict_with_generate=True,
        generation_max_length=MAX_LEN,
        fp16=True,
        optim="adamw_torch",
        logging_steps=100,
        report_to="none",
        seed=SEED,
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=tokenized,
        tokenizer=tokenizer,
        data_collator=DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100),
    )

    print(f"Fine-tuning {model_name} ...")
    trainer.train()

    os.makedirs(save_dir, exist_ok=True)
    contiguous = {k: v.contiguous() for k, v in model.state_dict().items()}
    model.save_pretrained(save_dir, state_dict=contiguous, safe_serialization=False)
    tokenizer.save_pretrained(save_dir)
    print(f"Saved to {save_dir}")

    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
train_generator(ARAT5_MODEL,   ARAT5_DIR,   train_df)
train_generator(ARABART_MODEL, ARABART_DIR, train_df)

## 6. Load both generators

In [ ]:
print("Loading fine-tuned AraT5v2 ...")
arat5_tokenizer = AutoTokenizer.from_pretrained(ARAT5_DIR)
arat5_model     = AutoModelForSeq2SeqLM.from_pretrained(ARAT5_DIR).to(DEVICE).eval()

print("Loading fine-tuned AraBART ...")
arabart_tokenizer = AutoTokenizer.from_pretrained(ARABART_DIR)
arabart_model     = AutoModelForSeq2SeqLM.from_pretrained(ARABART_DIR).to(DEVICE).eval()

if DEVICE == "cuda":
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Both generators loaded. VRAM {used:.1f} / {total:.1f} GB")

## 7. Propose and select

Each generator proposes `NUM_RETURN` rewrites; the pooled candidates are scored
on polarity, agreement with their peers, and overlap with the source; the
highest-scoring candidate wins.

In [ ]:
chrf_metric = CHRF()


def non_empty(prediction, fallback):
    """Guarantee a usable string, falling back to the source if decoding failed."""
    text = str(prediction).strip() if prediction is not None else ""
    return text if text else str(fallback)


def propose(model, tokenizer, prompts, sources):
    """Return NUM_RETURN candidate rewrites per prompt from one generator."""
    encoded = tokenizer(
        prompts, return_tensors="pt", max_length=MAX_LEN, truncation=True, padding=True
    ).to(DEVICE)
    with torch.no_grad():
        outputs = model.generate(
            **encoded,
            max_new_tokens=MAX_LEN,
            num_beams=NUM_BEAMS,
            num_return_sequences=NUM_RETURN,
            early_stopping=True,
            no_repeat_ngram_size=3,
            repetition_penalty=1.2,
            length_penalty=1.0,
        )
    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    cleaned = [non_empty(text, sources[i // NUM_RETURN]) for i, text in enumerate(decoded)]
    return [cleaned[i * NUM_RETURN:(i + 1) * NUM_RETURN] for i in range(len(prompts))]


def consensus(candidate, pool):
    """Mean chrF between a candidate and the rest of its pool.

    Minimum Bayes Risk in spirit: a candidate that most of the pool agrees with
    is more likely to be a genuine rewrite, whereas an outlier is usually a
    truncation, a repetition, or a hallucination.
    """
    others = [c for c in pool if c != candidate]
    if not others:
        return 1.0
    return float(np.mean([chrf_metric.sentence_score(candidate, [o]).score / 100.0
                          for o in others]))


def select(pool, source, target_polarity, pool_labels):
    """Pick the pool member that best balances polarity against content.

    The sentiment term is the classifier confidence, signed by whether the
    candidate landed on the target polarity, so a confidently wrong candidate is
    penalised harder than an uncertain one. The content term is chrF against the
    source, which is what makes a one-word edit preferable to a paraphrase.
    """
    best_text, best_score = pool[0], float("-inf")

    for candidate, prediction in zip(pool, pool_labels):
        hit = prediction["label"].capitalize() == target_polarity
        sentiment_score = prediction["score"] if hit else -prediction["score"]
        consensus_score = consensus(candidate, pool)
        content_score   = chrf_metric.sentence_score(candidate, [source]).score / 100.0

        score = (W_SENTIMENT * sentiment_score
                 + W_CONSENSUS * consensus_score
                 + W_CONTENT * content_score)

        if score > best_score:
            best_score, best_text = score, candidate

    return best_text


def predict(df, arat5, arat5_tok, arabart, arabart_tok):
    """Run the full propose-then-select pipeline over a dataframe."""
    predictions = []

    for start in range(0, len(df), GEN_BATCH):
        batch    = df.iloc[start:start + GEN_BATCH]
        prompts  = batch["input_text"].tolist()
        sources  = batch["source"].astype(str).tolist()
        targets  = batch["target_polarity"].tolist()

        pools = [a + b for a, b in zip(
            propose(arat5,   arat5_tok,   prompts, sources),
            propose(arabart, arabart_tok, prompts, sources),
        )]

        flat   = [c for pool in pools for c in pool]
        labels = classify_all(flat)

        cursor = 0
        for pool, source, target in zip(pools, sources, targets):
            pool_labels = labels[cursor:cursor + len(pool)]
            cursor += len(pool)
            predictions.append(select(pool, source, target, pool_labels))

        done = min(start + GEN_BATCH, len(df))
        print(f"  {done} / {len(df)}", end="\r")

    print(f"\nGenerated {len(predictions)} predictions.")
    return predictions

In [ ]:
predictions = predict(val_df, arat5_model, arat5_tokenizer,
                      arabart_model, arabart_tokenizer)

assert len(predictions) == len(val_df)
assert all(str(p).strip() for p in predictions)

## 8. Evaluation

Sentiment accuracy here is directly comparable to the official metric, since it
uses the same classifier and the gold source polarity. The BLEU and chrF figures
are computed against the reference target and are reported only as internal
diagnostics; they did not reproduce the official scorer exactly.

In [ ]:
bleu_metric = BLEU(effective_order=True)

references = val_df["target"].astype(str).tolist()
targets    = val_df["target_polarity"].tolist()

bleu_scores = [bleu_metric.sentence_score(p, [r]).score
               for p, r in zip(predictions, references)]
chrf_scores = [chrf_metric.sentence_score(p, [r]).score
               for p, r in zip(predictions, references)]

predicted_labels = classify_all(predictions)
correct = sum(p["label"].capitalize() == t
              for p, t in zip(predicted_labels, targets))

print("Development results")
print("-" * 46)
print(f"  Sentiment accuracy : {correct / len(predictions):.4f}  ({correct}/{len(predictions)})")
print(f"  BLEU  vs reference : {np.mean(bleu_scores):.2f}   [internal diagnostic]")
print(f"  chrF  vs reference : {np.mean(chrf_scores):.2f}   [internal diagnostic]")

## 9. Submission file

In [ ]:
def write_submission(ids, predictions, work_dir):
    """Write predictions.zip in the exact layout the official scorer expects.

    The scorer opens `predictions.xlsx` at the archive root. Passing `arcname`
    is what puts it there; zipping the path itself would nest it in directories
    and the scorer would not find it.
    """
    xlsx_path = f"{work_dir}/predictions.xlsx"
    zip_path  = f"{work_dir}/predictions.zip"

    submission = pd.DataFrame({"id": list(ids), "style": list(predictions)})
    submission.to_excel(xlsx_path, index=False)

    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        archive.write(xlsx_path, arcname="predictions.xlsx")

    # Verify by reading back exactly what the scorer will read.
    with zipfile.ZipFile(zip_path) as archive:
        assert archive.namelist() == ["predictions.xlsx"], archive.namelist()

    check = pd.read_excel(xlsx_path)
    assert list(check.columns) == ["id", "style"]
    assert len(check) == len(submission)
    assert check["id"].tolist() == list(ids)
    assert check["style"].notna().all()
    assert (check["style"].astype(str).str.strip() != "").all()

    print(f"Wrote {zip_path}  ({len(check)} rows)")
    return submission


submission = write_submission(val_df["id"], predictions, WORK_DIR)
submission.head()